In [29]:
# ── View aligned outputs in napari ─────────────────────────────────────────────
import napari
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from scipy.ndimage import maximum_filter, distance_transform_edt as edt_func
import numpy as np
import sknw
from skimage.morphology import skeletonize as skeletonize_3d
from scipy.ndimage import distance_transform_edt as edt
import sys
from pathlib import Path
import matplotlib.colors as mcolors
import colorsys
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

from skeletonisation import (prune_graph, graph2image, compute_cross_sectional_areas,
                              remove_mid_node, collect_border_vicinity_edges)

def custom_napari_palette(n_labels, base_colour, hue_span=0.02, sat_span=0.45, light_span=0.45, seed=0, alpha=1.0):
    """Build a varied Napari label colormap by jittering H/S/L around a base Matplotlib color."""
    n_labels = int(n_labels)
    if n_labels < 1:
        return {0: np.array([0.0, 0.0, 0.0, 0.0], dtype=float)}

    rng = np.random.default_rng(seed)
    r, g, b = mcolors.to_rgb(base_colour)
    base_h, base_l, base_s = colorsys.rgb_to_hls(r, g, b)

    hue = (base_h + rng.uniform(-hue_span, hue_span, n_labels)) % 1.0
    sat = np.clip(base_s + rng.uniform(-sat_span, sat_span, n_labels), 0.20, 1.00)
    light = np.clip(base_l + rng.uniform(-light_span, light_span, n_labels), 0.18, 0.88)

    order = rng.permutation(n_labels)
    custom_colormap = {0: np.array([0.0, 0.0, 0.0, 0.0], dtype=float)}
    for idx, j in enumerate(order, start=1):
        pr, pg, pb = colorsys.hls_to_rgb(float(hue[j]), float(light[j]), float(sat[j]))
        custom_colormap[idx] = np.array([pr, pg, pb, float(alpha)], dtype=float)
    return custom_colormap





def trim_segmentation(segmentation, fill_threshold = 0.75):
    slice_fill = segmentation.astype(bool).mean(axis=(1, 2))  # (Z,) fraction per slice
    keep_start = 0
    while keep_start < len(slice_fill) and slice_fill[keep_start] > fill_threshold:
        keep_start += 1
    keep_end = len(slice_fill) - 1
    while keep_end >= keep_start and slice_fill[keep_end] > fill_threshold:
        keep_end -= 1

    return segmentation [keep_start:keep_end+1]


In [17]:

root_dir  = Path(r"Z:\Bel\Vascumap_Outputs")
# out_dir   = root_dir / "Graphs"
# out_dir.mkdir(exist_ok=True)

subfolders = sorted([d for d in root_dir.iterdir() if d.is_dir() and d.name != "Graphs"])
print(f"Found {len(subfolders)} subfolders")

folder = subfolders[-6]
print(folder)

Found 58 subfolders
Z:\Bel\Vascumap_Outputs\marina_M4_2025.02.28_FL32_FL33_img4_Bead_Merged


In [ ]:

original = np.load(sorted(folder.glob("*cropped_stack_aligned.npy"))[0])
segmentation = np.load(sorted(folder.glob("*_clean_segmentation.npy"))[0])
translation = np.load(sorted(folder.glob("*_translation_aligned.npy"))[0])



In [20]:
viewer = napari.Viewer()
viewer.add_image(original)
viewer.add_image(translation)
viewer.add_labels(segmentation.astype(np.uint32))


<Labels layer 'Labels' at 0x1a3c966b090>

In [40]:
folder  = Path(r"Z:\Bel\Vascumap_Outputs\Dorota_merged_stellaris_Tulip10_Dorota_merged_stellaris_Tulip10")
segmentation = np.load(sorted(folder.glob("*_clean_segmentation.npy"))[0])
holes = np.load(sorted(folder.glob("*hole_labels_per_slice.npy"))[0])

In [41]:
viewer = napari.Viewer()

viewer.add_labels(
    segmentation.astype(np.uint32),
    name='clean_graph_skeleton',
    colormap=custom_napari_palette(1, 'red', hue_span=0, sat_span=0, light_span=0),
    opacity=1,
    )
viewer.add_labels(
    holes.astype(np.uint32),
    name='holes',
    colormap=custom_napari_palette(500, 'dodgerblue', hue_span=0, sat_span=0.4, light_span=0.4),
    opacity=1,
    )

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\skeletonise\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\skeletonise\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


<Labels layer 'holes' at 0x1a439d36390>

In [42]:
folder = Path("Z:\Bel\Vascumap_Outputs\Farid_18_07_25_img0_Valve 1")
segmentation = np.load(sorted(folder.glob("*_clean_segmentation.npy"))[0])

original = np.load(sorted(folder.glob("*cropped_stack_aligned.npy"))[0])

translation = np.load(sorted(folder.glob("*vessel_translation_aligned.npy"))[0])

In [43]:
viewer = napari.Viewer()

viewer.add_labels(
    segmentation.astype(np.uint32),
    name='clean_graph_skeleton',
    colormap=custom_napari_palette(1, 'red', hue_span=0, sat_span=0, light_span=0),
    opacity=1,
    )
viewer.add_image(translation)
viewer.add_image(original)

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\skeletonise\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


<Image layer 'original' at 0x1a366225910>